# 지도 시각화

In [1]:
#!pip install folium

### 기본 지도 그리기

In [50]:
import folium

m = folium.Map(
    location=[37.566826, 126.9786567],
    zoom_start=11
)

m.save('data/map_basic.html')
print('지도 저장 완료: data/map_basic.html')
m

지도 저장 완료: data/map_basic.html


### 마커 + 팝업 + 툴팁

In [6]:
locations = [
    {'name': '경복궁', 'lat': 37.5796, 'lon': 126.9770, 'desc': '조선의 법궁'},
    {'name': '남산타워', 'lat': 37.5512, 'lon': 126.9882, 'desc': 'N서울타워'},
    {'name': '인사동', 'lat': 37.5741, 'lon': 126.9862, 'desc': '전통문화거리'},
]

m2 = folium.Map(location=[37.566, 126.977], zoom_start=13)

for loc in locations:
    folium.Marker(
        location=[loc['lat'], loc['lon']],
        popup=folium.Popup(f"<b>{loc['name']}</b><br>{loc['desc']}", max_width=200),
        tooltip=loc['name'],
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(m2)

m2.save('data/map_markers.html')
m2

### Circle Marker

In [8]:
import pandas as pd

data = [
    ('강남구', 37.5172, 127.0473, 564000),
    ('서초구', 37.4837, 127.0324, 444000),
    ('종로구', 37.5730, 126.9794, 153000),
    ('송파구', 37.5145, 127.1058, 671000),
    ('강서구', 37.5509, 126.8495, 588000),
    ('노원구', 37.6541, 127.0568, 514000),
]
df = pd.DataFrame(data, columns=['구', '위도', '경도', '인구'])

m3 = folium.Map(location=[37.566, 126.978], zoom_start=11)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['위도'], row['경도']],
        radius=row['인구'] / 20000,
        color='crimson',
        fill=True,
        fill_color='crimson',
        fill_opacity=0.5,
        tooltip=f"{row['구']}: {row['인구']:,}명"
    ).add_to(m3)

m3.save('data/map_circle.html')
m3

### 지하철 2호선 혼잡도 히트맵

In [9]:
import folium
from folium.plugins import HeatMap

line2_stations = [
    ('강남', 37.4979, 127.0276, 200000),
    ('홍대입구', 37.5572, 126.9249, 190000),
    ('신촌', 37.5553, 126.9364, 160000),
    ('신림', 37.4846, 126.9290, 175000),
    ('구로디지털단지', 37.4854, 126.9012, 140000),
    ('잠실', 37.5133, 127.1001, 210000),
    ('왕십리', 37.5613, 127.0375, 130000),
    ('성수', 37.5445, 127.0556, 120000),
    ('서울대입구', 37.4812, 126.9528, 145000),
    ('사당', 37.4763, 126.9817, 180000),
]

m_heat = folium.Map(location=[37.530, 126.990], zoom_start=12)

# 히트맵 데이터: [위도, 경도, 가중치]
heat_data = [[s[1], s[2], s[3] / 210000] for s in line2_stations]

HeatMap(heat_data, radius=30, blur=20, max_zoom=13).add_to(m_heat)

# 역명 마커 추가
for name, lat, lon, cnt in line2_stations:
    folium.CircleMarker(
        location=[lat, lon], radius=6,
        color='darkblue', fill=True, fill_color='white',
        tooltip=f'{name}: {cnt:,}명'
    ).add_to(m_heat)

m_heat.save('data/map_subway_heat.html')
m_heat

### 따릉이 대여소 위치 표시 - 자치구별 거치대수 많은 기준 대여소 1개씩

In [3]:
import pandas as pd
import folium

df = pd.read_excel('data/공공자전거 대여소 정보(25.12월 기준).xlsx', header=None)
df.head(10)
df_bike = df.iloc[5:, [1, 2, 4, 5, 8]]
df_bike.head()

,1,2,4,5,8
5,망원역 1번출구 앞,마포구,37.555649,126.910629,15
6,망원역 2번출구 앞,마포구,37.554951,126.910835,14
7,합정역 1번출구 앞,마포구,37.550629,126.914986,13
8,합정역 5번출구 앞,마포구,37.550007,126.914825,5
9,합정역 7번출구 앞,마포구,37.548645,126.912827,12


In [4]:
# 컬럼명 수정
df_bike.columns = ['대여소명', '자치구','위도', '경도', '거치대수']
df_bike.head()

,대여소명,자치구,위도,경도,거치대수
5,망원역 1번출구 앞,마포구,37.555649,126.910629,15
6,망원역 2번출구 앞,마포구,37.554951,126.910835,14
7,합정역 1번출구 앞,마포구,37.550629,126.914986,13
8,합정역 5번출구 앞,마포구,37.550007,126.914825,5
9,합정역 7번출구 앞,마포구,37.548645,126.912827,12


In [10]:
# 자치구별 거치대수가 가장 많은 대여소 필터
st = df_bike.loc[ df_bike.groupby('자치구')['거치대수'].sum()]
st['위도'] = st['위도'].astype(float)
st['경도'] = st['경도'].astype(float)
st.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25 entries, 907 to 517
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   대여소명    25 non-null     object 
 1   자치구     25 non-null     object 
 2   위도      25 non-null     float64
 3   경도      25 non-null     float64
 4   거치대수    9 non-null      object 
dtypes: float64(2), object(3)
memory usage: 1.2+ KB


In [6]:
# 지도 생성
m_bike = folium.Map(location=[37.530, 126.990], zoom_start=12)

In [8]:
# 마커 생성
for idx, row in st.iterrows():
    folium.Marker(
        location=[row['위도'], row['경도']],
        popup=row['대여소명'],
        tooltip=row['대여소명']
    ).add_to(m_bike)

m_bike.save('data/map_bike.html')
m_bike